# Data Generator Application
Generate synthetic data using, Hugging Face, LLAMA, and Gradio

# Prerequisite installs

In [ ]:
!pip install -q --upgrade bitsandbytes accelerate transformers==4.57.6

# Setup
Imports, definitions, and login to hugging face


In [3]:

import tempfile

from attr import dataclass
from dotenv import load_dotenv
from huggingface_hub import login
import os
import gradio as gr
from transformers import AutoModelForCausalLM, AutoTokenizer, TextStreamer
from google.colab import userdata

LLAMA = "meta-llama/Llama-3.2-3B-Instruct"
SYSTEM_PROMPT = """"
You are a helpful assistant that generates synthetic data.
You will create realistic data based on the name of each column and the optional description.

You will output json with the generated data. Generate valid JSON only.
Do not generate any additional text.
"""

USER_PROMPT_TEMPLATE = """
Generate synthetic data for the following columns:
{columns}

Generate exactly {rows} rows of data.

Output the data as valid JSON.
Do not include any additional text outside the JSON.
"""
@dataclass
class Column:
    name: str
    description: str = ""

hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

# Create Functions
These functions will be used by the gradio app to call the LLM and generate the file

In [4]:
def generate_user_prompt(columns:list[Column], rows:int):
    columns_str = "\n".join([f"- {col.name}: {col.description}" for col in columns])
    return USER_PROMPT_TEMPLATE.format(columns=columns_str, rows=rows)

def get_model_output(model_name:str, messages:list):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token

    inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to("cuda")
    streamer = TextStreamer(tokenizer)

    model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")

    outputs = model.generate(inputs, streamer=streamer, max_new_tokens=4096)
    response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)

    return response

def generate_synthetic_data(columns:str, rows:int):
    columns_list:list[Column] = []
    for col in columns.split("\n"):
        if ":" in col:
            name, description = col.split(":", 1)
            columns_list.append(Column(name=name.strip(), description=description.strip()))
        else:
            columns_list.append(Column(name=col.strip()))

    user_prompt = generate_user_prompt(columns_list, int(rows))

    output = get_model_output(LLAMA, [{"role": "system", "content": "You are a helpful assistant"}, {"role": "user", "content": user_prompt}])

    json_data = output.find("```json")
    if json_data != -1:
        json_data = output[json_data + 7:]
        json_data = json_data[:json_data.find("```")]

        return json_data, create_json_file(json_data)

    return "{ \"error\": \"Failed to generate JSON\" }", None

def create_json_file(json_data: str) -> str:
    fd, path = tempfile.mkstemp(suffix=".json")
    with os.fdopen(fd, "w") as tmp:
        tmp.write(json_data)

    return path

# Create the Gradio App

In [5]:
def create_gradio_app():
    with gr.Blocks() as app:
        with gr.Row():
            gr.Markdown("""
            # Synthetic Data Generator
            Enter the column names and descriptions, and specify the number of rows to generate synthetic data.
            """)
        with gr.Row():
            gr.Markdown("""
            ## Column list
            Enter the column names and descriptions below, one per line.
            Use the format `Column Name: Description` for each line.
            """)
            num_rows = gr.Number(label="Number of Rows", value=10)
        with gr.Row(scale=5):
            columns = gr.TextArea(label="Column Names and Descriptions", placeholder="Enter column names and descriptions here...")
        with gr.Row():
            with gr.Column(scale=2):
                pass
            with gr.Column(scale=1):
                generate_button = gr.Button("Generate Synthetic Data")
            with gr.Column(scale=2):
                pass
        with gr.Row():
            output = gr.TextArea(label="Generated Synthetic Data")
        with gr.Row():
            with gr.Column(scale=2):
                pass
            with gr.Column(scale = 1):
                file_output = gr.File(label="Download Generated Data")
            with gr.Column(scale=2):
                pass

        generate_button.click(
            fn=generate_synthetic_data,
            inputs=[columns, num_rows],
            outputs=[output, file_output]
        )

        return app

# Run the App

In [ ]:
app = create_gradio_app()
app.launch()